## New `gwfast` examples

* The `Signal` code is largely rewritten
  * The `GWstrain` now accepts a dictionary of keys, only the necessary ones will do.
  * Some repetitive functions, such as antenna pattern and geocentric time delay are separated.
  * The `signal_derivative`, Fisher codes, and `WFOverlap` rely entirely on the `GWstrain` function, so a child class simply needs to redefine the `GWstrain`, then everything else will just work.
* Added a `Detector` class
* The `FisherMatr` is now supporting:
  * Derivatives w.r.t. the input parameters
  * Maintaining the input order
  * Multi-dimensional inputs
* Updated `PhenomHM` to support multi-dimensional input
* Minor bugfix in the `lu` decomposition

In [1]:
from pathlib import Path

from jax import config
import jax.numpy as np
config.update("jax_enable_x64", True)

import numpy as onp

import gwfast.network as network
import gwfast.waveforms as waveforms
from gwfast.gwfastUtils import m1m2_from_Mceta
from gwfast.gwfastGlobals import detectors as det_dict, detPath
from gwfast.detector import Detector
from gwfast.signals import BasicGWSignal, AGNLensedGWSignal
import gwfast.fisherTools as fTools

/users/hin-wai.leong/src/AGN-gwfast/gwfast/waveforms.py:31: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


TEOBResumS is not installed, only the GWFAST waveform models are available, namely: TaylorF2, IMRPhenomD, IMRPhenomD_NRTidalv2, IMRPhenomHM and IMRPhenomNSBH


## The new `Signal` code

In [2]:
# Initialise the detector objects themselves
H1 = Detector('H1', **det_dict['H1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
L1 = Detector('L1', **det_dict['L1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
V1 = Detector('V1', **det_dict['Virgo'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/avirgo_O5low_NEW.txt')

# Define some waveforms
PhenomD = waveforms.IMRPhenomD()
PhenomHM = waveforms.IMRPhenomHM()

# Define the detectors and networks with vanilla BBH signal
H1_BBH = BasicGWSignal(wf_model=PhenomD, detector=H1, fmin=10)
L1_BBH = BasicGWSignal(wf_model=PhenomD, detector=L1, fmin=10)
L1_BBH_HM = BasicGWSignal(wf_model=PhenomHM, detector=L1, fmin=10)
V1_BBH = BasicGWSignal(wf_model=PhenomD, detector=V1, fmin=10)
HL_BBH_signals = network.DetNet({'H1': H1_BBH, 'L1': L1_BBH})
HLV_BBH_signals = network.DetNet({'H1': H1_BBH, 'L1': L1_BBH, 'V1': V1_BBH})

Initializing jax...
Jax local device count: 1
Jax device count: 1
Initializing jax...
Jax local device count: 1
Jax device count: 1
Initializing jax...
Jax local device count: 1
Jax device count: 1
Initializing jax...
Jax local device count: 1
Jax device count: 1


In [3]:
# Define the parameters
size = 10
parameters = {
    'Mc': 30, 'eta': 0.2, 'dL': 2, 'theta': 2.34, 'phi': 5.43,
    'iota': 0.99*np.pi/2, 'psi': 1, 'tGPS': 0, 'phase': 2.8,
    'chi1z': 1e-3, 'chi2z': 1e-3,
}
# This casting is to ensure all int becomes float
parameters = {key: np.full(size, val).astype(np.float64) for key, val in parameters.items()}
# Define the frequency array to be evaluated on
N_freqs = 500
f_array = np.geomspace(20, 400, num=N_freqs)
# Broadcast the frequency array to the appropriate shape for multiple parameter values
f_array = np.broadcast_to(f_array, (parameters['Mc'].shape[0], N_freqs)).T

L1_BBH_strains = L1_BBH.GWstrain(f_array, parameters)
L1_BBH_strains.shape

(500, 10)

### Implicit Transformations

In [4]:
parameters_2  = parameters.copy()
Mc = parameters_2.pop('Mc')
eta = parameters_2.pop('eta')
theta = parameters_2.pop('theta')
phi = parameters_2.pop('phi')
# Convert chirp mass, mass ratio to component masses
m1, m2 = m1m2_from_Mceta(Mc, eta)
parameters_2['m1'] = m1
parameters_2['m2'] = m2
parameters_2['ra'] = phi
parameters_2['dec'] = np.pi/2 - theta  
print(parameters_2.keys())

# This casting is to ensure all int becomes float
parameters_2 = {key: np.full(size, val).astype(np.float64) for key, val in parameters_2.items()}

L1_BBH_strains_2 = L1_BBH.GWstrain(f_array, parameters_2)
L1_BBH_strains_2.shape

dict_keys(['dL', 'iota', 'psi', 'tGPS', 'phase', 'chi1z', 'chi2z', 'm1', 'm2', 'ra', 'dec'])


(500, 10)

In [5]:
print(f'Strain RMS: {np.sqrt(np.mean(np.abs(L1_BBH_strains[:, 0])**2)):.4e}')  # Check the strain amplitude
np.allclose(
    L1_BBH_strains[:, 0], L1_BBH_strains_2[:, 0], atol=1e-40, rtol=1e-15)  # Check both methods yield the same result

Strain RMS: 6.0306e-25


Array(True, dtype=bool)

## The Fisher code

In [6]:
fisher_mats_BBH = HLV_BBH_signals.FisherMatr(parameters)
cov_mats, ie = fTools.CovMatr(fisher_mats_BBH)
fTools.print_single_matrix(cov_mats[:, :, 0], parameters)

Computing Fisher for H1...
Computing Fisher for L1...
Computing Fisher for V1...
Done.
             Mc           eta          dL          theta         phi         iota          psi         tGPS         phase        chi1z        chi2z   
    Mc   +3.651e+00   -1.129e-01   +7.129e-02   +2.123e-03   +1.870e-03   +1.353e-03   -4.454e-03   -1.715e-01   -2.142e+01   +4.309e+00   -1.111e+01  
   eta   -1.129e-01   +7.937e-03   +1.095e-02   -1.929e-04   -2.848e-04   -2.059e-04   +4.784e-04   +9.764e-03   +8.677e-01   -2.437e-01   +6.414e-01  
    dL   +7.129e-02   +1.095e-02   +1.460e-01   -1.707e-02   -2.608e-02   +2.295e-03   +5.073e-02   +9.766e-03   +2.291e-01   -2.417e-01   +6.641e-01  
 theta   +2.123e-03   -1.929e-04   -1.707e-02   +8.214e-03   +1.081e-02   -2.768e-03   -2.413e-02   -1.914e-04   -1.929e-02   +5.474e-03   -1.451e-02  
   phi   +1.870e-03   -2.848e-04   -2.608e-02   +1.081e-02   +1.853e-02   -1.255e-03   -3.565e-02   -2.588e-04   -2.414e-02   +7.759e-03   -2.093e-02  
  

In [7]:
fisher_mats_BBH_2 = HLV_BBH_signals.FisherMatr(parameters_2)
cov_mats_2, ie = fTools.CovMatr(fisher_mats_BBH_2)
fTools.print_single_matrix(cov_mats_2[:, :, 0], parameters_2)

Computing Fisher for H1...
Computing Fisher for L1...
Computing Fisher for V1...
Done.
             dL          iota          psi         tGPS         phase        chi1z        chi2z         m1           m2           ra           dec    
    dL   +1.460e-01   +2.295e-03   +5.073e-02   +9.766e-03   +2.291e-01   -2.417e-01   +6.641e-01   -3.666e+00   +1.265e+00   -2.608e-02   +1.707e-02  
  iota   +2.295e-03   +1.333e-02   +5.957e-03   -2.883e-04   +1.708e-03   +7.202e-03   -1.871e-02   +7.407e-02   -2.184e-02   -1.255e-03   +2.768e-03  
   psi   +5.073e-02   +5.957e-03   +8.536e-02   +4.399e-04   +4.322e-02   -1.340e-02   +3.574e-02   -1.746e-01   +4.981e-02   -3.565e-02   +2.413e-02  
  tGPS   +9.766e-03   -2.883e-04   +4.399e-04   +1.349e-02   +1.257e+00   -3.326e-01   +8.860e-01   -3.716e+00   +9.579e-01   -2.588e-04   +1.914e-04  
 phase   +2.291e-01   +1.708e-03   +4.322e-02   +1.257e+00   +1.403e+02   -3.124e+01   +8.210e+01   -3.420e+02   +8.064e+01   -2.414e-02   +1.929e-02  
 c

In [8]:
for key, val in zip(parameters.keys(), onp.sqrt(onp.diag(cov_mats[:, :, 0]))):
    print(key, '--', val)

for key, val in zip(parameters_2.keys(), onp.sqrt(onp.diag(cov_mats_2[:, :, 0]))):
    print(key, '--', val)

Mc -- 1.9108882861830026546
eta -- 0.08908908440342910012
dL -- 0.38214532276966703202
theta -- 0.09063366214125003453
phi -- 0.13611723688279611556
iota -- 0.115455706794577287686
psi -- 0.29216277066144086255
tGPS -- 0.11615645620018702297
phase -- 11.845202171656193262
chi1z -- 2.8672885263614236594
chi2z -- 7.6292213185828507333
dL -- 0.38214532277045435148
iota -- 0.11545570679457918084
psi -- 0.2921627706614417413
tGPS -- 0.11615645620689913362
phase -- 11.845202172303719692
chi1z -- 2.867288526530024902
chi2z -- 7.629221319020975774
m1 -- 33.454225372953681424
m2 -- 9.01642369393040612
ra -- 0.13611723688279695668
dec -- 0.0906336621412510968


## Multi-dimensional input

In [9]:
# PhenomHM

# Define the parameters
size = (3, 3, 2)
parameters = {
    'Mc': 30, 'eta': 0.2, 'dL': 2, 'theta': 2.34, 'phi': 5.43,
    'iota': 0.99*np.pi/2, 'psi': 1, 'tGPS': 0, 'phase': 2.8,
    'chi1z': 1e-3, 'chi2z': 1e-3,
}
# This casting is to ensure all int becomes float
parameters = {key: np.full(size, val).astype(np.float64) for key, val in parameters.items()}
# Define the frequency array to be evaluated on
N_freqs = 500
minimum_frequency_array = np.full(size, 20)
maximum_frequency_array = np.full(size, 400)
f_array = np.geomspace(minimum_frequency_array, maximum_frequency_array, num=N_freqs)

L1_BBH_strains = L1_BBH_HM.GWstrain(f_array, parameters)
L1_BBH_strains.shape

(500, 3, 3, 2)

In [10]:
# Fisher

fisher_mats_BBH = HLV_BBH_signals.FisherMatr(parameters)
cov_mats, ie = fTools.CovMatr(fisher_mats_BBH)
fTools.print_single_matrix(cov_mats[:, :, 0, 0, 0], parameters)
fisher_mats_BBH.shape

Computing Fisher for H1...
Computing Fisher for L1...
Computing Fisher for V1...
Done.
             Mc           eta          dL          theta         phi         iota          psi         tGPS         phase        chi1z        chi2z   
    Mc   +3.651e+00   -1.129e-01   +7.129e-02   +2.123e-03   +1.870e-03   +1.353e-03   -4.454e-03   -1.715e-01   -2.142e+01   +4.309e+00   -1.111e+01  
   eta   -1.129e-01   +7.937e-03   +1.095e-02   -1.929e-04   -2.848e-04   -2.059e-04   +4.784e-04   +9.764e-03   +8.677e-01   -2.437e-01   +6.414e-01  
    dL   +7.129e-02   +1.095e-02   +1.460e-01   -1.707e-02   -2.608e-02   +2.295e-03   +5.073e-02   +9.766e-03   +2.291e-01   -2.417e-01   +6.641e-01  
 theta   +2.123e-03   -1.929e-04   -1.707e-02   +8.214e-03   +1.081e-02   -2.768e-03   -2.413e-02   -1.914e-04   -1.929e-02   +5.474e-03   -1.451e-02  
   phi   +1.870e-03   -2.848e-04   -2.608e-02   +1.081e-02   +1.853e-02   -1.255e-03   -3.565e-02   -2.588e-04   -2.414e-02   +7.759e-03   -2.093e-02  
  

(11, 11, 3, 3, 2)